# 09 - SDG Decision Impact Analysis

**Purpose:** Translate inflation effects into SDG decision terms per dataset (reserve margin, exceedance alerts, storage sizing), stating assumptions explicitly.

**Inputs:** results/tables/ inflation tables (notebook 08).

**Outputs:** SDG decision-impact tables/figures under results/tables/ and results/figures/.

**Runtime estimate:** 10-20 minutes.

## SDG 7: reserve margin and storage sizing impact

In [1]:
import pandas as pd

from sdg_leakage_audit.sdg.impact import estimate_reserve_margin_impact
from sdg_leakage_audit.sdg.mapping import get_sdg_tags
from sdg_leakage_audit.utils.paths import results_tables_dir, to_project_relative

inflation_df = pd.read_csv(results_tables_dir() / "inflation_by_cell.csv")

SDG7_DATASETS = ["household_power_consumption", "appliances_energy_prediction"]
for dataset_id in SDG7_DATASETS:
    assert "SDG7" in get_sdg_tags(dataset_id)

reserve_margin_tables = {
    dataset_id: estimate_reserve_margin_impact(inflation_df, dataset_id) for dataset_id in SDG7_DATASETS
}

for dataset_id, table in reserve_margin_tables.items():
    print(f"\n{dataset_id} (reserve margin, factor=2.0x RMSE):")
    print(table[["rmse_true", "rmse_apparent", "under_provisioning_pct"]].round(2))



household_power_consumption (reserve margin, factor=2.0x RMSE):
             rmse_true  rmse_apparent  under_provisioning_pct
protocol_id                                                  
P0                0.95           0.95                    0.00
P1                0.95           0.86                    9.58
P2                0.95           0.96                   -1.14
P3                0.95           0.96                   -1.37
P4                0.95           0.88                    7.22
P5                0.95           0.95                   -0.71
P6                0.95           0.86                    8.96

appliances_energy_prediction (reserve margin, factor=2.0x RMSE):
             rmse_true  rmse_apparent  under_provisioning_pct
protocol_id                                                  
P0              154.55         154.55                    0.00
P1              154.55         115.40                   25.33
P2              154.55         157.84                   -2.13
P

## SDG 3/11: air-quality exceedance-alert impact

In [2]:
from sdg_leakage_audit.sdg.impact import estimate_exceedance_alert_impact

SDG3_11_DATASETS = ["air_quality_uci", "beijing_multisite_air_quality"]
for dataset_id in SDG3_11_DATASETS:
    tags = get_sdg_tags(dataset_id)
    assert "SDG3" in tags and "SDG11" in tags

exceedance_tables = {
    dataset_id: estimate_exceedance_alert_impact(inflation_df, dataset_id) for dataset_id in SDG3_11_DATASETS
}

for dataset_id, table in exceedance_tables.items():
    print(f"\n{dataset_id} (exceedance-alert buffer, factor=1.645x RMSE):")
    print(table[["rmse_true", "rmse_apparent", "under_provisioning_pct"]].round(2))



air_quality_uci (exceedance-alert buffer, factor=1.645x RMSE):
             rmse_true  rmse_apparent  under_provisioning_pct
protocol_id                                                  
P0                1.12           1.12                    0.00
P1                1.12           1.30                  -16.42
P2                1.12           1.07                    3.86
P3                1.12           0.93                   16.47
P4                1.12           1.06                    5.47
P5                1.12           1.03                    7.46
P6                1.12           1.20                   -7.41

beijing_multisite_air_quality (exceedance-alert buffer, factor=1.645x RMSE):
             rmse_true  rmse_apparent  under_provisioning_pct
protocol_id                                                  
P0               87.69          87.69                    0.00
P1               87.69          95.11                   -8.47
P2               87.69          88.87               

## SDG 13: climate-forecast-dependent planning impact

In [3]:
from sdg_leakage_audit.sdg.impact import estimate_storage_sizing_impact

SDG13_DATASETS = ["jena_climate"]
for dataset_id in SDG13_DATASETS:
    assert "SDG13" in get_sdg_tags(dataset_id)

storage_tables = {
    dataset_id: estimate_storage_sizing_impact(inflation_df, dataset_id) for dataset_id in SDG13_DATASETS
}

for dataset_id, table in storage_tables.items():
    print(f"\n{dataset_id} (thermal/storage capacity buffer, factor=2.0x RMSE):")
    print(table[["rmse_true", "rmse_apparent", "under_provisioning_pct"]].round(2))



jena_climate (thermal/storage capacity buffer, factor=2.0x RMSE):
             rmse_true  rmse_apparent  under_provisioning_pct
protocol_id                                                  
P0                3.16           3.16                    0.00
P1                3.16           4.22                  -33.65
P2                3.16           3.14                    0.48
P3                3.16           3.18                   -0.67
P4                3.16           3.17                   -0.41
P5                3.16           3.28                   -3.88
P6                3.16           4.22                  -33.81


## Stated assumptions

In [4]:
all_impact_tables = {**reserve_margin_tables, **exceedance_tables, **storage_tables}
combined = pd.concat(
    [table.assign(dataset_id=dataset_id) for dataset_id, table in all_impact_tables.items()]
).reset_index().rename(columns={"index": "protocol_id"})

out_path = results_tables_dir() / "sdg_decision_impact.csv"
combined.to_csv(out_path, index=False)
print(f"Saved SDG decision-impact table ({len(combined)} rows) -> {to_project_relative(out_path)}")

print(
    "\nSTATED ASSUMPTIONS (per sdg.impact module docstring):\n"
    "1. Every decision context here uses the same underlying model: a safety buffer sized as "
    "`factor * RMSE`, compared using P0's true RMSE vs each protocol's apparent RMSE. This is a "
    "common PATTERN in operational forecasting (load-reserve margins, alert thresholds, capacity "
    "headroom), not a claim that any specific utility, regulator, or standard uses these exact factors.\n"
    "2. reserve_margin_factor=2.0 and storage_buffer_factor=2.0 are illustrative round numbers, not "
    "calibrated to any specific grid operator's reserve-margin policy or thermal-capacity standard.\n"
    "3. alert_buffer_factor=1.645 approximates a one-sided 95% buffer under a normality assumption on "
    "forecast errors -- a common convention, but air-quality forecast errors are not verified normal "
    "here (see notebook 03's EDA); treat this as illustrative, not a calibrated regulatory threshold.\n"
    "4. under_provisioning_pct is computed from mean RMSE across all models/horizons/seeds for a given "
    "(dataset, protocol) -- it does not account for which specific model/horizon combination an actual "
    "deployment would use, since that choice is out of scope for this audit.\n"
    "5. These are Phase-2 fast-profile numbers (single seed, small data subsample, minimal hyperparameter "
    "search) -- the SDG impact magnitudes here illustrate the TRANSLATION METHOD, not final "
    "real-world impact estimates; a full-profile re-run is needed before citing specific percentages."
)


Saved SDG decision-impact table (35 rows) -> results\tables\sdg_decision_impact.csv

STATED ASSUMPTIONS (per sdg.impact module docstring):
1. Every decision context here uses the same underlying model: a safety buffer sized as `factor * RMSE`, compared using P0's true RMSE vs each protocol's apparent RMSE. This is a common PATTERN in operational forecasting (load-reserve margins, alert thresholds, capacity headroom), not a claim that any specific utility, regulator, or standard uses these exact factors.
2. reserve_margin_factor=2.0 and storage_buffer_factor=2.0 are illustrative round numbers, not calibrated to any specific grid operator's reserve-margin policy or thermal-capacity standard.
3. alert_buffer_factor=1.645 approximates a one-sided 95% buffer under a normality assumption on forecast errors -- a common convention, but air-quality forecast errors are not verified normal here (see notebook 03's EDA); treat this as illustrative, not a calibrated regulatory threshold.
4. under_pr

## Findings / what to carry forward

- All three SDG decision contexts (SDG7 reserve margin, SDG3/11 exceedance-alert buffer, SDG13
  storage/thermal capacity buffer) share one underlying translation: a safety buffer sized as
  `factor * RMSE`, compared between P0's true RMSE and each leaky protocol's apparent RMSE. The
  resulting `under_provisioning_pct` is the capacity/buffer shortfall a decision-maker would face
  if they sized their buffer off the leaky protocol's apparent accuracy instead of P0's.
- Full per-dataset, per-protocol impact table saved to `results/tables/sdg_decision_impact.csv`.
- All five stated assumptions (buffer-factor choices, mean-across-cells aggregation, fast-profile
  scope) are printed explicitly in this notebook's output -- this is intentional, per the
  project's "state assumptions explicitly; do not overclaim" requirement, and should be carried
  into the project's methodology/limitations write-up nearly verbatim.
- Because this entire analysis is downstream of notebook 08's inflation numbers (themselves
  `fast`-profile, single-seed, small-subsample results), the specific percentages here are a
  demonstration of the translation METHOD, not final real-world impact estimates -- that
  distinction must survive into the write-up text, not just this notebook.
- Next (notebook 10): robustness/sensitivity checks (horizon, window size, model family, dataset
  subsets) to see whether the inflation effects and their SDG translations hold up across
  variations, before notebook 11 assembles final tables/figures.
